In [15]:
%matplotlib inline
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [16]:
import numpy as np
import numpy.random as npr
import pandas as pd
import matplotlib.pyplot as plt
import ssm
from sklearn import preprocessing
from sklearn.model_selection import KFold
from scipy import stats

from notebooks.imports import *
from config import dir_config, main_config
from src.utils.glm_hmm_utils import *
from src.utils.glm_hmm_utils_cv import *

import pickle
import copy

import numpy as np
import numpy.random as npr
import pandas as pd
from sklearn import preprocessing

### Configuration

In [17]:
from config import dir_config, main_config

raw_dir = Path(dir_config.data.raw)
processed_dir = Path(dir_config.data.processed)
glm_hmm_dir = Path(processed_dir, "glm_hmm_models")

glm_hmm_config = main_config["GLM_HMM"]

metadata = pd.read_csv(Path(processed_dir, "processed_metadata_accu_60.csv"))
data = pd.read_csv(Path(processed_dir, "processed_data_accu_60_all.csv"))

### Load and Prepare data

In [18]:
experiment_sites = ["Stanford", "Harvard"]

metadata = metadata[metadata['experiment_site'].isin(experiment_sites)].reset_index(drop=True)
data = data[data['subject_id'].isin(metadata['subject_id'])].reset_index(drop=True)

# add session_id to data with matching subject_id and medication
data.choice = data.choice.fillna(-1).astype(int)
data.target = data.target.fillna(-1).astype(int)
data.outcome = data.outcome.fillna(-1).astype(int)


In [19]:
off_session_ids = data[data["medication"] == "off"].session_id.unique()
on_session_ids = data[data["medication"] == "on"].session_id.unique()
off_session_ids.sort()
on_session_ids.sort()

### Data processing

In [20]:
print("------------- info ----------------")
print(data.info())
print("------------- Head ----------------")
print(data.head())
print("\n\n------------- describe ----------------\n\n")
print(data.describe())
print("------------- nan counts ----------------")
print(data.isnull().sum())
print("\n\n------------- dtypes ----------------\n\n")
print(data.dtypes)
print("\n\n------------- shape ----------------\n\n")
print(data.shape)

------------- info ----------------
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 28401 entries, 0 to 28400
Data columns (total 15 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   subject_id        28401 non-null  object 
 1   medication        28401 non-null  object 
 2   prior             28401 non-null  object 
 3   prior_direction   28401 non-null  object 
 4   prior_color       28401 non-null  object 
 5   color             28045 non-null  float64
 6   coherence         28045 non-null  float64
 7   target            28401 non-null  int64  
 8   is_valid          28401 non-null  bool   
 9   outcome           28401 non-null  int64  
 10  choice            28401 non-null  int64  
 11  reaction_time     28401 non-null  float64
 12  session_filename  28401 non-null  object 
 13  signed_coherence  28045 non-null  float64
 14  session_id        28401 non-null  object 
dtypes: bool(1), float64(4), int64(3), object(7)
memory 

#### Data preparation

#### Create design matrix (input, output, mask)

In [21]:
CURRENT_TRIAL_FEATURES = glm_hmm_config["current_trial_features"] + ["bias"] if glm_hmm_config["add_bias"] else glm_hmm_config["current_trial_features"]
PREV_TRIAL_FEATURES = glm_hmm_config["prev_trial_features"]
N_TRIALS_BACK = glm_hmm_config["n_trials_back"]

MODEL_FEATURES = CURRENT_TRIAL_FEATURES + [f"{var}_{n + 1}" for n in range(N_TRIALS_BACK) for var in PREV_TRIAL_FEATURES]
glm_hmm_config["model_features"] = MODEL_FEATURES

INPUT_DIM = len(MODEL_FEATURES)
MODEL_FEATURES


['normalized_stimulus', 'bias', 'prev_choice_1', 'prev_coherence_1', 'prev_choice_coherence_1']

In [22]:
def extract_previous_trial_data(session_data, valid_idx, first_trial):
    """
    Extracts previous trial features for each trial in session_data.
    Returns a dict mapping feature names to arrays of shape (n_trials, N_TRIALS_BACK).
    """
    n_trials = session_data.shape[0] - first_trial
    prev_data = {}

    # Precompute numpy arrays for speed
    signed_coherence = (session_data.signed_coherence.values * 2) - 1
    color = (session_data.color.values * 2) - 1
    target = (session_data.target.values * 2) - 1
    choice = (session_data.choice.values * 2) - 1
    outcome = session_data.outcome.values

    # Initialize output arrays
    for var in PREV_TRIAL_FEATURES:
        prev_data[var] = np.empty((n_trials, N_TRIALS_BACK), dtype=int)

    for i in range(first_trial, session_data.shape[0]):
        valid_indices = valid_idx[valid_idx < i][-N_TRIALS_BACK:]
        for idx_var, var in enumerate(PREV_TRIAL_FEATURES):
            var_col = var[5:] if var.startswith("prev_") else var
            padded_indices = np.pad(valid_indices, (N_TRIALS_BACK - len(valid_indices), 0), "constant", constant_values=0)
            if var_col == "coherence":
                vals = signed_coherence
            elif var_col == "choice_outcome":
                vals = choice * outcome
            elif var_col == "choice_coherence":
                vals = choice * signed_coherence
            elif var_col == "coherence_color_outcome":
                vals = color * outcome * signed_coherence
            else:
                vals = session_data[var_col].values

            prev_data[var][i - first_trial] = vals[padded_indices]
    return prev_data

def prepare_input_data(data, valid_idx, first_trial):
    n_trials = data.shape[0] - first_trial
    X = np.ones((1, n_trials, INPUT_DIM))
    # Fill current trial features
    for idx, feat in enumerate(CURRENT_TRIAL_FEATURES):
        if feat == "normalized_stimulus":
            X[0, :, idx] = data.signed_coherence.values[first_trial:] / 100
        elif feat == "bias":
            X[0, :, idx] = 1
        elif feat == "stimulus_color":
            X[0, :, idx] = (data.signed_coherence.values[first_trial:]/100) * (data.color.values[first_trial:])
        else:
            X[0, :, idx] = data[feat].values[first_trial:]
    # Fill previous trial features
    prev_data = extract_previous_trial_data(data, valid_idx, first_trial)
    col_idx = len(CURRENT_TRIAL_FEATURES)
    for var in PREV_TRIAL_FEATURES:
        for n in range(N_TRIALS_BACK):
            X[0, :, col_idx] = prev_data[var][:, n]
            col_idx += 1
    return list(X)


def process_sessions(data, session_ids):
    inputs_session_wise = []
    choices_session_wise = []
    invalid_idx_session_wise = []
    masks_session_wise = []
    reaction_time_session_wise = []
    color_session_wise = []

    for session_id in session_ids:
        session_data = data[data["session_id"] == session_id]

        valid_idx = np.where(session_data.outcome >= 0)[0]

        # First valid trial considering n_trial_back
        first_trial = valid_idx[N_TRIALS_BACK - 1] + 1

        # Prepare inputs
        inputs = prepare_input_data(session_data, valid_idx, first_trial)

        # Keep choices 1D
        choices = session_data.choice.values[first_trial:].astype(int)
        # Compute invalid_idx relative to first_trial
        invalid_idx = np.where(session_data.outcome.values[first_trial:] < 0)[0]

        # Replace invalid trials with random 0/1
        if invalid_idx.size > 0:
            choices[invalid_idx] = np.random.choice([0, 1], size=invalid_idx.size)

        # Create mask: True everywhere, False at invalid_idx
        mask = np.ones_like(choices, dtype=bool)
        mask[invalid_idx] = False

        # Reaction times
        reaction_times = session_data.reaction_time.values[first_trial:]
        reaction_time_session_wise.append(reaction_times.reshape(-1, 1))

        # color
        color = session_data.color.values[first_trial:]
        color_session_wise.append(color.reshape(-1, 1))

        # Save session data
        masks_session_wise.append(mask.reshape(-1, 1))
        choices_session_wise.append(choices.reshape(-1, 1))
        inputs_session_wise += inputs

    # Normalize inputs (except bias) for all sessions
    unnormalized_inputs_session_wise = copy.deepcopy(inputs_session_wise)
    indices_without_bias = np.flatnonzero(np.array(glm_hmm_config["model_features"]) != "bias")
    for idx_session in range(len(inputs_session_wise)):
        mask = masks_session_wise[idx_session][:, 0].astype(bool)
        row_idx, col_idx = np.where(mask)[0], indices_without_bias
        # select rows where mask is True and only non-bias columns
        X = inputs_session_wise[idx_session][np.ix_(row_idx, col_idx)]
        # scale those columns
        X_scaled = preprocessing.scale(X, axis=0)
        # put them back
        inputs_session_wise[idx_session][np.ix_(row_idx, col_idx)] = X_scaled

    models_glm_hmm, fit_lls_glm_hmm = global_fit(choices_session_wise, inputs_session_wise, state_range=np.arange(1, 6), masks=masks_session_wise, n_iters=2500, n_initializations=20)

    # get best model of 20 initializations for each state
    init_params = {"glm_weights": {}, "transition_matrices": {}}
    for n_states in np.arange(1, 6):
        best_idx = fit_lls_glm_hmm[n_states].index(max(fit_lls_glm_hmm[n_states]))
        init_params["glm_weights"][n_states] = models_glm_hmm[n_states][best_idx].observations.params
        init_params["transition_matrices"][n_states] = models_glm_hmm[n_states][best_idx].transitions.params

    # session-wise fitting with 5 fold cross-validation
    models_session_state_fold, train_ll_session, test_ll_session = session_wise_fit_cv(
        choices_session_wise, inputs_session_wise, masks=masks_session_wise, n_sessions=len(session_ids), init_params=init_params, state_range=np.arange(1, 6), n_iters=1000
    )

    # store data and models for aggregated
    global_fits = {"models": models_glm_hmm, "fits_lls_glm_hmm": fit_lls_glm_hmm, "init_params": init_params}
    session_wise_fits = {
        "models": models_session_state_fold,
        "train_ll": train_ll_session,
        "test_ll": test_ll_session,
    }

    # store data and models for session-wise
    session_data = {}
    for idx, session_id in enumerate(session_ids):
        inputs = inputs_session_wise[idx]
        df = {
            "choices": choices_session_wise[idx].ravel(),
            "stimulus": unnormalized_inputs_session_wise[idx][:, 0],
            "mask": masks_session_wise[idx].ravel(),
            "color": color_session_wise[idx].ravel(),
            "reaction_time": reaction_time_session_wise[idx].ravel(),
        }

        for i, feat in enumerate(MODEL_FEATURES):
            df[feat] = inputs[:, i]

        session_data[session_id] = pd.DataFrame(df)

    models_and_data = {
        "global": global_fits,
        "session_wise": session_wise_fits,
        "data": session_data,
    }
    return models_and_data

In [23]:
model_dict = {"config": glm_hmm_config}

#### On med sessions

In [24]:
on_meds = process_sessions(data=data, session_ids=on_session_ids)
model_dict["on_meds"] = on_meds

Fitting GLM globally...


  0%|          | 0/2500 [00:00<?, ?it/s]

Fitting 1 states...


Converged to LP: -7326.7:   0%|          | 2/2500 [00:00<02:36, 15.93it/s]


Fitting 2 states...


...

LP: -383.2:   0%|          | 0/1000 [00:00<?, ?it/s]

Fitting 4 states...


LP: -423.8:   0%|          | 0/1000 [00:00<?, ?it/s]

Fitting 5 states...


Converged to LP: -400.8:  17%|█▋        | 166/1000 [00:01<00:09, 88.26it/s]


#### Off med sessions

In [25]:
off_meds = process_sessions(data=data, session_ids=off_session_ids)
model_dict["off_meds"] = off_meds

Fitting GLM globally...


  0%|          | 0/2500 [00:00<?, ?it/s]

Fitting 1 states...


LP: -7416.1:   0%|          | 1/2500 [00:00<05:12,  8.00it/s]

Fitting 2 states...


...

LP: -363.4:   0%|          | 5/1000 [00:00<00:21, 46.24it/s]

Fitting 4 states...


Converged to LP: -359.0:  17%|█▋        | 168/1000 [00:01<00:07, 108.53it/s]


Fitting 5 states...


Converged to LP: -392.5:   9%|▊         | 87/1000 [00:01<00:10, 84.73it/s]


In [26]:
with open(f"{Path(glm_hmm_dir, glm_hmm_config['name'])}.pkl", "wb") as f:
    pickle.dump(model_dict, f)

In [27]:
model_dict.keys()

dict_keys(['config', 'on_meds', 'off_meds'])